# Explainability

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yazanjer/An_Explainable_AI_Education/blob/main/notebooks/06_explainability.ipynb)

**Answers:** Editor comments 7 and 8
**Estimated runtime:** 1 h · **Hardware:** CPU
**Quick mode:** set `QUICK_MODE = True` in the setup cell for a fast smoke test.

Local SHAP, global SHAP and LIME computed, labelled and reported **separately**.
Backgrounds come from training data only. LIME is repeated with different seeds and
its stability reported. `noise_control` is appended as a negative control.

Every interpreted variable is resolved through the official PISA codebook; no
educational meaning is assigned without a documentary source.

---


In [ ]:
# --- Environment setup -------------------------------------------------
# Detects Colab, mounts Drive only when in Colab, installs pinned deps.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
QUICK_MODE = True   # set False for the full budget

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT = Path("/content/drive/MyDrive/An_Explainable_AI_Education")
    PROJECT.mkdir(parents=True, exist_ok=True)
    if not (PROJECT / "src").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/yazanjer/An_Explainable_AI_Education.git", str(PROJECT)],
                       check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT / "requirements.txt")], check=False)
else:
    PROJECT = Path(os.environ.get("VLPSO_PROJECT_ROOT", Path.cwd().parent))

os.environ["VLPSO_PROJECT_ROOT"] = str(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))

from vlpso_xai.config import load_config, set_global_seeds, environment_report
cfg = load_config("quick" if QUICK_MODE else "default")
set_global_seeds(cfg.seed)
cfg.paths.mkdirs()
print("project root:", cfg.paths.root)
print("config:", cfg.config_path.name, "| hash:", cfg.hash()[:12])


In [ ]:
# --- LIME on the SAME feature space and the SAME fitted estimator -------
from vlpso_xai.explain.lime_local import select_instances, lime_with_stability
from vlpso_xai.explain.consistency import rank_agreement, CAUSAL_CAVEAT

scores = clf.predict_proba(Xte_t)[:, 1]
pick = select_instances(scores, yte, n=cfg.section("explain","lime","n_instances"),
                        seed=cfg.seed)
print("instance selection rule:", pick["rule"])

lime_out = lime_with_stability(
    clf, Xtr_t, Xte_t.iloc[pick["index"]],
    n_repeats=cfg.section("explain","lime","n_repeats"),
    n_samples=cfg.section("explain","lime","n_samples"),
    seed=cfg.seed)
display(lime_out["stability"])

# SHAP and LIME now share an estimator AND a feature space, so this comparison
# is meaningful. Previously SHAP saw the bare classifier and LIME the pipeline.
print(rank_agreement(imp, lime_out["weights"]))
print("\n" + CAUSAL_CAVEAT)

In [ ]:
# --- Global SHAP, on the CORRECT feature space --------------------------
# Audit finding C3: an earlier version passed the bare classifier RAW frames
# while it had been fitted on imputed/scaled/indicator-augmented data, and
# split students at random rather than by school. Both are fixed here.
import numpy as np
from vlpso_xai.data.design import school_grouped_splitter, assert_group_disjoint
from vlpso_xai.explain.shap_global import (global_shap, transformed_frames,
                                           noise_control_benchmark)
from vlpso_xai.models.pipeline import make_pipeline
from vlpso_xai.models.registry import get_models

# SCHOOL-GROUPED split, not train_test_split: classmates must not straddle it.
tr, te = next(iter(school_grouped_splitter(3, shuffle=True, random_state=cfg.seed)
                   .split(X, y, groups=g)))
assert_group_disjoint(tr, te, g)
Xtr, Xte, ytr, yte = X.iloc[tr], X.iloc[te], y[tr], y[te]

pipe = make_pipeline(
    get_models(["RandomForest"], fast=QUICK_MODE)["RandomForest"]["model"]).fit(Xtr, ytr)

# Push both frames through the FITTED preprocessing so the explainer sees the
# same feature space the estimator was fitted on.
Xtr_t, Xte_t = transformed_frames(pipe, Xtr, Xte)
clf = pipe.named_steps["clf"]
assert Xte_t.shape[1] == clf.n_features_in_

gs = global_shap(clf, Xtr_t, Xte_t, yte, model_name="RandomForest",
                 n_instances=cfg.section("explain", "global_shap", "n_instances"),
                 background_size=cfg.section("explain", "global_shap", "background_size"),
                 seed=cfg.seed)
imp = gs["importance"]

# Report real items separately from imputation artefacts (audit finding M6).
imp["is_indicator"] = imp.feature.str.startswith("missingindicator_")
imp["codebook_label"] = [
    "(missing-data indicator)" if i else cb.label(f)
    for f, i in zip(imp.feature, imp.is_indicator)]
display(imp[~imp.is_indicator].head(15))
print(f"{imp.is_indicator.sum()} of {len(imp)} columns are missing-data indicators, "
      "excluded from the substantive feature count.")
display(pd.DataFrame([gs["spec"].as_row()]).T)

In [ ]:
# --- LIME on the SAME feature space and the SAME fitted estimator -------
from vlpso_xai.explain.lime_local import select_instances, lime_with_stability
from vlpso_xai.explain.consistency import rank_agreement, CAUSAL_CAVEAT

scores = clf.predict_proba(Xte_t)[:, 1]
pick = select_instances(scores, yte, n=cfg.section("explain","lime","n_instances"),
                        seed=cfg.seed)
print("instance selection rule:", pick["rule"])

lime_out = lime_with_stability(
    clf, Xtr_t, Xte_t.iloc[pick["index"]],
    n_repeats=cfg.section("explain","lime","n_repeats"),
    n_samples=cfg.section("explain","lime","n_samples"),
    seed=cfg.seed)
display(lime_out["stability"])

# SHAP and LIME now share an estimator AND a feature space, so this comparison
# is meaningful. Previously SHAP saw the bare classifier and LIME the pipeline.
print(rank_agreement(imp, lime_out["weights"]))
print("\n" + CAUSAL_CAVEAT)